In [1]:
!nvidia-smi

Wed Sep  2 07:50:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!unzip -o "/content/material (4).zip" -d /content/
!ls /content

Archive:  /content/material (4).zip
  inflating: /content/model-lock.md  
  inflating: /content/smoke_test.py  
  inflating: /content/verify_cell.py  
'material (4).zip'   sample_data     verify_cell.py
 model-lock.md	     smoke_test.py


In [3]:
import subprocess, sys

VLLM_PIN = "0.6.*"
BITSANDBYTES_PIN = "0.49.2"
AUTOAWQ_PIN = "0.2.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

In [4]:
import sys
print(sys.version)

!nvidia-smi --query-gpu=name --format=csv,noheader

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Tesla T4


In [5]:
pip_install(
    f"vllm=={VLLM_PIN}",
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"autoawq=={AUTOAWQ_PIN}",
    f"httpx=={HTTPX_PIN}",
    f"openai=={OPENAI_PIN}",
)

print("serving + AWQ pins installed")

installing: vllm==0.6.* transformers==4.46.* accelerate==1.1.* autoawq==0.2.* httpx==0.27.* openai==1.54.*
serving + AWQ pins installed


In [6]:
import os, signal, subprocess

MODEL = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"
PORT = 8000
SERVER_LOG = "/content/server.log"

SERVER_ARGS = {
    "--model": MODEL,
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(PORT),
    "--quantization": "awq",
    "--enable-auto-tool-choice": None,
    "--tool-call-parser": "hermes",
}

def build_cmd(args: dict) -> list:
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server"]

    for k, v in args.items():
        if v is None:
            cmd.append(k)
        else:
            cmd += [k, str(v)]

    return cmd

def launch_server(args: dict = None):
    args = SERVER_ARGS if args is None else args
    cmd = build_cmd(args)

    print("launching:", " ".join(cmd))

    logf = open(SERVER_LOG, "wb")

    proc = subprocess.Popen(
        cmd,
        stdout=logf,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server()

launching: /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --quantization awq --enable-auto-tool-choice --tool-call-parser hermes
server pid 2663, logging to /content/server.log


In [7]:
import time, urllib.request, urllib.error

def tail_log(path=SERVER_LOG, n=30):
    try:
        with open(path, "r", errors="replace") as fh:
            lines = fh.readlines()
        return "".join(lines[-n:])
    except FileNotFoundError:
        return "(no log file yet)"


def wait_for_health(port=PORT, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s

    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(
                        f"server healthy after about {waited}s: "
                        f"{url} -> 200"
                    )
                    return True

        except (urllib.error.URLError, ConnectionError, OSError):
            pass

        time.sleep(interval_s)

    print(f"TIMED OUT after {timeout_s}s waiting for {url}")
    print("last 30 log lines:")
    print(tail_log())

    return False


healthy = wait_for_health()

server healthy after about 155s: http://localhost:8000/v1/models -> 200


In [8]:
print(tail_log(n=50))

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:53<00:00,  1.53s/it]
INFO 09-02 08:01:19 model_runner.py:1535] Graph capturing finished in 54 secs, took 0.31 GiB
INFO 09-02 08:01:19 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 62.57 seconds
INFO 09-02 08:01:23 api_server.py:640] Using supplied chat template:
INFO 09-02 08:01:23 api_server.py:640] None
INFO 09-02 08:01:23 serving_chat.py:73] "auto" tool choice has been enabled please note that while the parallel_tool_calls client option is preset for compatibility reasons, it will be ignored.
INFO 09-02 08:01:23 launcher.py:19] Available routes are:
INFO 09-02 08:01:23 launcher.py:27] Route: /openapi.json, Methods: GET, HEAD
INFO 09-02 08:01:23 launcher.py:27] Route: /docs, Methods: GET, HEAD
INFO 09-02 08:01:23 launcher.py:27] Route: /docs/oauth2-redirect, Methods: GET, HEAD
INFO 09-02 08:01:23 launcher.py:27] Route: /redoc, Methods: GET, HEAD
INFO:     Started server process [2663]
INFO:     Wai

In [9]:
!grep -i "GPU blocks" /content/server.log

INFO 09-02 08:00:19 gpu_executor.py:76] # GPU blocks: 22955, # CPU blocks: 9362


In [10]:
SPOT_PROMPTS = [
    "Write a two-sentence summary of what an inference server does.",

    "A user asks for the weather in Riyadh and the time in Tokyo. "
    "What two tool calls would you make?",

    "Refactor this into a single sentence: The GPU was busy but not "
    "productive, because decode is memory-bound.",

    "List the steps to roll back a bad deployment, in order.",

    "Explain quantisation to a non-technical manager in three sentences.",
]

from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed"
)

for p in SPOT_PROMPTS:
    r = client.chat.completions.create(
        model="Qwen/Qwen2.5-1.5B-Instruct-AWQ",
        messages=[
            {"role": "user", "content": p}
        ],
        max_tokens=200
    )

    print("=" * 70)
    print("PROMPT:", p)
    print()
    print(r.choices[0].message.content)
    print()

PROMPT: Write a two-sentence summary of what an inference server does.

An inference server is responsible for processing incoming requests and generating responses based on the input data, typically for tasks such as machine learning model inference, predictive analytics, and automated decision-making.

PROMPT: A user asks for the weather in Riyadh and the time in Tokyo. What two tool calls would you make?

To fetch the weather in Riyadh and the time in Tokyo for a specific date, you would typically make two API calls. Assuming the services that provide these information are accessible through two different APIs (API One for weather and API Two for time), you would issue the following calls:

For weather in Riyadh:
```plaintext
http://api.weather.com/w/chat?apiKey=[your_api_key]&conditions=all&urls=http://weather.com/re/ (replaced "re" with Riyadh)
```

To fetch the time in Tokyo (`Tokyo`):
```plaintext
http://api.timeZone.com/timetables/v1/timeships.json?key=apikey&cityIds=891&api-la

No obvious degradation across the five prompts.

In [11]:
from smoke_test import run_smoke

result_awq = run_smoke(
    base_url="http://localhost:8000/v1",
    model="Qwen/Qwen2.5-1.5B-Instruct-AWQ"
)

print(result_awq)

{'model': 'Qwen/Qwen2.5-1.5B-Instruct-AWQ', 'total_attempts': 10, 'score': 10, 'distractor_attempts': 2, 'distractor_call_free': 2, 'distractor_majority_clean': True, 'per_prompt': {'two_tool': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'single': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'distractor': {'k': 2, 'wants_call': False, 'valid': 0, 'call_free': 2}}, 'passed': True}


In [12]:
import urllib.request

try:
    r = urllib.request.urlopen(
        "http://localhost:8000/v1/models",
        timeout=5
    )
    print("Server status:", r.status)
    print(r.read().decode()[:500])
except Exception as e:
    print("ERROR:", e)

Server status: 200
{"object":"list","data":[{"id":"Qwen/Qwen2.5-1.5B-Instruct-AWQ","object":"model","created":1788336231,"owned_by":"vllm","root":"Qwen/Qwen2.5-1.5B-Instruct-AWQ","parent":null,"max_model_len":4096,"permission":[{"id":"modelperm-344a3fd7b5fa4dd7ac7712acff1a3343","object":"model_permission","created":1788336231,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":


In [13]:
def shutdown_server(proc=None, port=PORT):
    try:
        proc = server if proc is None else proc

        os.killpg(
            os.getpgid(proc.pid),
            signal.SIGTERM
        )

        print(
            f"sent SIGTERM to process group "
            f"of pid {proc.pid}"
        )

    except (ProcessLookupError, NameError):
        print("no server process to kill")

    time.sleep(3)

    try:
        with urllib.request.urlopen(
            f"http://localhost:{port}/v1/models",
            timeout=2
        ):
            print(
                f"WARNING: port {port} still answering"
            )

    except (
        urllib.error.URLError,
        ConnectionError,
        OSError
    ):
        print(f"port {port} is free")


shutdown_server()

sent SIGTERM to process group of pid 2663
port 8000 is free


In [14]:
SERVER_ARGS = {
    "--model": "Qwen/Qwen2.5-1.5B-Instruct",
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": "8000",
    "--enable-auto-tool-choice": None,
    "--tool-call-parser": "hermes",
}

server = launch_server(SERVER_ARGS)

launching: /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --enable-auto-tool-choice --tool-call-parser hermes
server pid 4016, logging to /content/server.log


In [15]:
healthy = wait_for_health()

TIMED OUT after 300s waiting for http://localhost:8000/v1/models
last 30 log lines:
INFO 09-02 08:05:45 model_runner.py:1415] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_utilization` or switching to eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.

Capturing CUDA graph shapes:  77%|███████▋  | 27/35 [07:01<10:03, 75.40s/it] 


In [18]:
!nvidia-smi --query-gpu=memory.used --format=csv,noheader

11565 MiB


In [19]:
for p in SPOT_PROMPTS:
    r = client.chat.completions.create(
        model="Qwen/Qwen2.5-1.5B-Instruct",
        messages=[
            {"role": "user", "content": p}
        ],
        max_tokens=200
    )

    print("=" * 70)
    print("PROMPT:", p)
    print()
    print(r.choices[0].message.content)
    print()

PROMPT: Write a two-sentence summary of what an inference server does.

An inference server is responsible for processing incoming requests to execute model predictions and returning results to the client seamlessly and efficiently.

PROMPT: A user asks for the weather in Riyadh and the time in Tokyo. What two tool calls would you make?

To find both the weather in Riyadh and the time in Tokyo, you would need to use two tool calls:

1. For the weather, you would make a tool call to "WeatherForecast".
2. For the time, you would make a tool call to "TimeIn" or "Time".

Here's an example of how you might structure the JSON for these two tool calls using a hypothetical JSON web API or a chatbot API provided by the service:

```json
{
    "tool": "WeatherForecast",
    "parameters": {
        "city": "Riyadh"
    },
    "_request_id": "123456789"
},
{
    "tool": "Time",
    "parameters": {
        "location": "Tokyo",
        "number_of_decimals": 2
    },
    "_request_id": "987654321"
}


In [20]:
result_fp16 = run_smoke(
    base_url="http://localhost:8000/v1",
    model="Qwen/Qwen2.5-1.5B-Instruct"
)

print(result_fp16)

{'model': 'Qwen/Qwen2.5-1.5B-Instruct', 'total_attempts': 10, 'score': 10, 'distractor_attempts': 2, 'distractor_call_free': 2, 'distractor_majority_clean': True, 'per_prompt': {'two_tool': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'single': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'distractor': {'k': 2, 'wants_call': False, 'valid': 0, 'call_free': 2}}, 'passed': True}


In [21]:
result = result_awq
LOCKED_MODEL = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"

print("LOCKED MODEL:", LOCKED_MODEL)

LOCKED MODEL: Qwen/Qwen2.5-1.5B-Instruct-AWQ


In [22]:
import json

with open("smoke_result.json", "w") as f:
    json.dump(result, f, indent=2)

print("saved smoke_result.json")

saved smoke_result.json


In [23]:
!cat smoke_result.json

{
  "model": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
  "total_attempts": 10,
  "score": 10,
  "distractor_attempts": 2,
  "distractor_call_free": 2,
  "distractor_majority_clean": true,
  "per_prompt": {
    "two_tool": {
      "k": 4,
      "wants_call": true,
      "valid": 4,
      "call_free": 0
    },
    "single": {
      "k": 4,
      "wants_call": true,
      "valid": 4,
      "call_free": 0
    },
    "distractor": {
      "k": 2,
      "wants_call": false,
      "valid": 0,
      "call_free": 2
    }
  },
  "passed": true
}

In [26]:
awq = result_awq if "result_awq" in globals() else result

score = awq["score"]
distractor = "yes" if awq["distractor_majority_clean"] else "no"
passed = "yes" if awq["passed"] else "no"

if "result_fp16" in globals():
    measured = f"both - AWQ {score}/10, fp16 {result_fp16['score']}/10"
else:
    measured = f"AWQ - {score}/10"

lines = [
    "# Model lock (team record)",
    "",
    "## The locked model",
    "",
    "- Model id: `Qwen/Qwen2.5-1.5B-Instruct-AWQ`",
    "- Quantisation: `awq`",
    "- Why this one: Passed the smoke test and the quality spot check showed no obvious degradation.",
    "",
    "## The launch flags",
    "",
    "--model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096",
    "--gpu-memory-utilization 0.85 --quantization awq",
    "--enable-auto-tool-choice --tool-call-parser hermes",
    "",
    "- Tool-call parser: `hermes`",
    "",
    "## The smoke score",
    "",
    f"- Score (valid behaviours out of 10): `{score}`",
    f"- Distractor stayed call-free in the majority: `{distractor}`",
    f"- Passed the gate (>= 8/10 and distractor majority clean): `{passed}`",
    f"- Measured against: `{measured}`",
    "",
    "## Quality spot check note",
    "",
    "- The AWQ build held up across the five prompts with no obvious degradation."
]

with open("model-lock.md", "w") as f:
    f.write("\n".join(lines))

print("✅ model-lock.md created")
print("Score:", score)
print("Passed:", passed)

✅ model-lock.md created
Score: 10
Passed: yes


In [27]:
!cat model-lock.md

# Model lock (team record)

## The locked model

- Model id: `Qwen/Qwen2.5-1.5B-Instruct-AWQ`
- Quantisation: `awq`
- Why this one: Passed the smoke test and the quality spot check showed no obvious degradation.

## The launch flags

--model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096
--gpu-memory-utilization 0.85 --quantization awq
--enable-auto-tool-choice --tool-call-parser hermes

- Tool-call parser: `hermes`

## The smoke score

- Score (valid behaviours out of 10): `10`
- Distractor stayed call-free in the majority: `yes`
- Passed the gate (>= 8/10 and distractor majority clean): `yes`
- Measured against: `both - AWQ 10/10, fp16 10/10`

## Quality spot check note

- The AWQ build held up across the five prompts with no obvious degradation.

In [28]:
!ls -l smoke_result.json model-lock.md

-rw-rw-rw- 1 root root 776 Sep  2 08:20 model-lock.md
-rw-r--r-- 1 root root 533 Sep  2 08:17 smoke_result.json


In [29]:
exec(open("verify_cell.py").read())

smoke score: 10/10, distractor clean: True
model-lock.md: all fields filled
GREEN CHECK: PASS


In [30]:
shutdown_server()

sent SIGTERM to process group of pid 4016
port 8000 is free


In [31]:
exec(open("verify_cell.py").read())

smoke score: 10/10, distractor clean: True
model-lock.md: all fields filled
GREEN CHECK: PASS


In [32]:
from google.colab import files

for f_ in ["smoke_result.json", "model-lock.md"]:
    files.download(f_)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>